In [1]:
import os
import sys
import torch

# =====================================================================
# 1. RESOLUCIÓN DINÁMICA DE RUTAS (Git-friendly)
# =====================================================================
# os.getcwd() saca la ruta de la carpeta donde está el notebook.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..\.."))

# Añadimos la raíz al path de Python de esta sesión temporalmente.
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

#print(f"Buscando en: {PROJECT_ROOT}")
#print(f"Contenido de esa carpeta: {os.listdir(PROJECT_ROOT)}")

# =====================================================================
# 2. IMPORTACIONES (Ahora funcionan en cualquier máquina)
# =====================================================================
from src.pipelines.pipelineEval import run_pipeline 
from src.DataProvider.FifDataProvider import FifDataProvider
from src.model_interface.MiRepNetInterface import MiRepNetInterface
from src.epoch_processing.EpochProcessorPipeline import EpochProcessorPipeline
from src.epoch_processing.EuclideanAlignment import EuclideanAlignment
from src.epoch_processing.SpatialInterpolator import SpatialInterpolator
from raw_processing.RawProcessorPipeline import RawProcessorPipeline
from raw_processing.BandpassFilter import BandpassFilter
from raw_processing.NotchFilter import NotchFilter
from raw_processing.Resampler import Resampler
from raw_processing.CARReference import CARReference
from raw_processing.ICAProcessor import ICAProcessor
from raw_processing.AnnotationRenamer import AnnotationRenamer

LABEL_MAP = {
    "IZQUIERDA": "left_hand",
    "DERECHA":   "right_hand",
    "ABAJO":     "feet",
    "DESCANSO":  "rest",
}

# =====================================================================
# 3. CONFIGURACIÓN DE RUTAS Y MODELO RELATIVAS AL ROOT
# =====================================================================
WEIGHT_PATH = os.path.join(PROJECT_ROOT, "src", "pretrainedModels", "MIRepNet", "weight", "MIRepNet.pth")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Construimos las rutas de los fifs dinámicamente
fif_names = [
    os.path.join(PROJECT_ROOT, "EEG_controller_app", "recordings", f"suj2_{i}_raw.fif") for i in range(1, 7)
]

c:\Users\Miguel\.conda\envs\bci-mi-tfg\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import itertools

# 1. Definimos los filtros que queremos combinar (¡SIN el Renamer!)
filtros_opcionales = [
    NotchFilter(50.0),
    BandpassFilter(8.0, 30.0),
    CARReference(),
    Resampler(250),
    ICAProcessor()
]

# 2. Generamos TODAS las combinaciones posibles (desde 0 filtros hasta los 5)
pipelines_a_probar = []

for longitud in range(len(filtros_opcionales) + 1):
    for combinacion in itertools.combinations(filtros_opcionales, longitud):
        
        # Convertimos la tupla de la combinación a una lista
        lista_filtros = list(combinacion)
        
        # AÑADIMOS EL FILTRO OBLIGATORIO AL FINAL (o al principio, según tu diseño)
        lista_filtros.append(AnnotationRenamer(LABEL_MAP))
        
        # Creamos el objeto pipeline y lo guardamos
        pipeline_instancia = RawProcessorPipeline(lista_filtros)
        pipelines_a_probar.append(pipeline_instancia)

print(f"Se han generado {len(pipelines_a_probar)} pipelines distintos para evaluar.")

Se han generado 32 pipelines distintos para evaluar.


In [3]:

# =====================================================================
# 4. EJECUCIÓN DEL PIPELINE PERSONALIZADO
# =====================================================================
modelos = []

for i, pipeline in enumerate(pipelines_a_probar):
    print(f"\n--- Evaluando Pipeline {i+1}/{len(pipelines_a_probar)} ---")
    
    dataProvider = FifDataProvider(fif_paths=fif_names, raw_pipeline=pipeline, annotations_names=["left_hand", "right_hand"])
    modelo = MiRepNetInterface(device=device, weight_path=WEIGHT_PATH, training_clases=["left_hand", "right_hand"])
    modelos.append(modelo)


--- Evaluando Pipeline 1/32 ---
Loaded 108/110 parameters from pretrained model

--- Evaluando Pipeline 2/32 ---
Loaded 108/110 parameters from pretrained model

--- Evaluando Pipeline 3/32 ---
Loaded 108/110 parameters from pretrained model

--- Evaluando Pipeline 4/32 ---
Loaded 108/110 parameters from pretrained model

--- Evaluando Pipeline 5/32 ---
Loaded 108/110 parameters from pretrained model

--- Evaluando Pipeline 6/32 ---
Loaded 108/110 parameters from pretrained model

--- Evaluando Pipeline 7/32 ---
Loaded 108/110 parameters from pretrained model

--- Evaluando Pipeline 8/32 ---
Loaded 108/110 parameters from pretrained model

--- Evaluando Pipeline 9/32 ---
Loaded 108/110 parameters from pretrained model

--- Evaluando Pipeline 10/32 ---
Loaded 108/110 parameters from pretrained model

--- Evaluando Pipeline 11/32 ---
Loaded 108/110 parameters from pretrained model

--- Evaluando Pipeline 12/32 ---
Loaded 108/110 parameters from pretrained model

--- Evaluando Pipeline 1

In [5]:
import pandas as pd

resultados = []

for i, pipeline_raw in enumerate(pipelines_a_probar):
    print(f"\n--- Evaluando Pipeline {i+1}/{len(pipelines_a_probar)} ---")
    
    # Extraemos el nombre de los filtros para la tabla
    # (Ajusta '_processors' si tu clase RawProcessorPipeline usa otro nombre interno para la lista)
    nombres_filtros = " + ".join([type(f).__name__ for f in pipeline_raw._processors]) if hasattr(pipeline_raw, '_processors') else f"Combinación {i+1}"
    print(f"Filtros Raw: {nombres_filtros}")

    # ====================================================================
    # PASO 1: Inyectamos el pipeline RAW al DataProvider
    # ====================================================================
    dataProvider = FifDataProvider(fif_paths=fif_names, raw_pipeline=pipeline_raw, annotations_names=["left_hand", "right_hand"])
    
    # ====================================================================
    # PASO 2: Creamos el pipeline de EPOCHS usando los canales del DataProvider
    # ====================================================================
    mi_pipeline_epochs = EpochProcessorPipeline([
        EuclideanAlignment(), 
        SpatialInterpolator(actual_channel_positions=dataProvider.get_channel_names()), 
    ])
    
    modelo = MiRepNetInterface(device=device, weight_path=WEIGHT_PATH, training_clases=["left_hand", "right_hand"])
    
    # ====================================================================
    # PASO 3: Ejecutamos cada cosa en su lugar
    # ====================================================================
    metricas = run_pipeline(
        dataProvider=dataProvider, 
        model_interface=modelo, 
        epochs=10, 
        epoch_pipeline=mi_pipeline_epochs, # <--- AQUÍ VA EL DE EPOCHS
        validation_split=0.2,
        exclude_training_classes=None, 
        rename_training_classes=None,
        show_plots=False
    )
    
    # Guardamos los resultados
    metricas["Filtros Usados"] = nombres_filtros
    resultados.append(metricas)

# ==========================================
# CREAMOS LA TABLA RESUMEN FINAL
# ==========================================
df_resultados = pd.DataFrame(resultados)

# Si la función run_pipeline nos ha devuelto un diccionario con Accuracy, etc., ordenamos:
columnas_deseadas = ["Filtros Usados", "Accuracy", "Precision", "Recall", "F1-Score"]
# Filtramos solo las columnas que existan por si tu run_pipeline devolvió menos cosas
columnas_existentes = [col for col in columnas_deseadas if col in df_resultados.columns]

df_resultados = df_resultados[columnas_existentes]
df_resultados = df_resultados.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

display(df_resultados)


--- Evaluando Pipeline 1/32 ---
Filtros Raw: AnnotationRenamer
Loaded 108/110 parameters from pretrained model
Cargando c:\Users\Miguel\Documents\GitHub\TFG-BCI\EEG_controller_app\recordings\suj2_1_raw.fif ...
Cargando c:\Users\Miguel\Documents\GitHub\TFG-BCI\EEG_controller_app\recordings\suj2_2_raw.fif ...
Cargando c:\Users\Miguel\Documents\GitHub\TFG-BCI\EEG_controller_app\recordings\suj2_3_raw.fif ...
Cargando c:\Users\Miguel\Documents\GitHub\TFG-BCI\EEG_controller_app\recordings\suj2_4_raw.fif ...
Cargando c:\Users\Miguel\Documents\GitHub\TFG-BCI\EEG_controller_app\recordings\suj2_5_raw.fif ...
Cargando c:\Users\Miguel\Documents\GitHub\TFG-BCI\EEG_controller_app\recordings\suj2_6_raw.fif ...


c:\Users\Miguel\Documents\GitHub\TFG-BCI\src\DataProvider\FifDataProvider.py:95: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  combined_epochs = mne.concatenate_epochs(all_epochs_list)


Epoch   1/10 | train_loss=0.8950 train_acc=49.0% | val_loss=1.8129 val_acc=50.0% | lr=0.000976
Epoch   2/10 | train_loss=0.7732 train_acc=52.1% | val_loss=1.2312 val_acc=50.0% | lr=0.000905
Epoch   3/10 | train_loss=0.7134 train_acc=58.3% | val_loss=0.8936 val_acc=41.7% | lr=0.000794
Epoch   4/10 | train_loss=0.7182 train_acc=51.0% | val_loss=0.8483 val_acc=41.7% | lr=0.000655
Epoch   5/10 | train_loss=0.7254 train_acc=43.8% | val_loss=0.8226 val_acc=50.0% | lr=0.000500
Epoch   6/10 | train_loss=0.6853 train_acc=55.2% | val_loss=0.8090 val_acc=45.8% | lr=0.000345
Epoch   7/10 | train_loss=0.6892 train_acc=60.4% | val_loss=0.8022 val_acc=54.2% | lr=0.000206
Epoch   8/10 | train_loss=0.6832 train_acc=56.2% | val_loss=0.7966 val_acc=50.0% | lr=0.000095
Epoch   9/10 | train_loss=0.6847 train_acc=60.4% | val_loss=0.7957 val_acc=50.0% | lr=0.000024
Epoch  10/10 | train_loss=0.6846 train_acc=51.0% | val_loss=0.7931 val_acc=50.0% | lr=0.000000

--- Evaluando Pipeline 2/32 ---
Filtros Raw: Notc

c:\Users\Miguel\Documents\GitHub\TFG-BCI\src\DataProvider\FifDataProvider.py:95: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  combined_epochs = mne.concatenate_epochs(all_epochs_list)


Epoch   1/10 | train_loss=0.9113 train_acc=45.8% | val_loss=2.8779 val_acc=50.0% | lr=0.000976
Epoch   2/10 | train_loss=0.7260 train_acc=51.0% | val_loss=2.0393 val_acc=50.0% | lr=0.000905
Epoch   3/10 | train_loss=0.7489 train_acc=51.0% | val_loss=1.3365 val_acc=50.0% | lr=0.000794
Epoch   4/10 | train_loss=0.6850 train_acc=54.2% | val_loss=0.9932 val_acc=50.0% | lr=0.000655
Epoch   5/10 | train_loss=0.7129 train_acc=50.0% | val_loss=0.9647 val_acc=50.0% | lr=0.000500
Epoch   6/10 | train_loss=0.6998 train_acc=52.1% | val_loss=0.9899 val_acc=50.0% | lr=0.000345
Epoch   7/10 | train_loss=0.6885 train_acc=57.3% | val_loss=0.9775 val_acc=50.0% | lr=0.000206
Epoch   8/10 | train_loss=0.6816 train_acc=57.3% | val_loss=0.9586 val_acc=50.0% | lr=0.000095
Epoch   9/10 | train_loss=0.6834 train_acc=58.3% | val_loss=0.9506 val_acc=50.0% | lr=0.000024
Epoch  10/10 | train_loss=0.6745 train_acc=60.4% | val_loss=0.9376 val_acc=50.0% | lr=0.000000

--- Evaluando Pipeline 3/32 ---
Filtros Raw: Band

...

c:\Users\Miguel\Documents\GitHub\TFG-BCI\src\DataProvider\FifDataProvider.py:95: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  combined_epochs = mne.concatenate_epochs(all_epochs_list)
c:\Users\Miguel\Documents\GitHub\TFG-BCI\src\pretrainedModels\MiRepNet\utils\utils.py:75: ComplexWarning: Casting complex values to real discards the imaginary part
  XEA[i] = np.dot(sqrtRefEA, x[i])


Epoch   1/10 | train_loss=0.8098 train_acc=42.7% | val_loss=0.6931 val_acc=41.7% | lr=0.000976
Epoch   2/10 | train_loss=0.7019 train_acc=54.2% | val_loss=0.7870 val_acc=50.0% | lr=0.000905
Epoch   3/10 | train_loss=0.6708 train_acc=51.0% | val_loss=0.7653 val_acc=50.0% | lr=0.000794
Epoch   4/10 | train_loss=0.6428 train_acc=61.5% | val_loss=0.8264 val_acc=50.0% | lr=0.000655
Epoch   5/10 | train_loss=0.6331 train_acc=64.6% | val_loss=1.0937 val_acc=50.0% | lr=0.000500
Epoch   6/10 | train_loss=0.6037 train_acc=69.8% | val_loss=1.4210 val_acc=50.0% | lr=0.000345
Epoch   7/10 | train_loss=0.5841 train_acc=69.8% | val_loss=1.6516 val_acc=50.0% | lr=0.000206
Epoch   8/10 | train_loss=0.5972 train_acc=71.9% | val_loss=1.8047 val_acc=50.0% | lr=0.000095
Epoch   9/10 | train_loss=0.5623 train_acc=72.9% | val_loss=1.8802 val_acc=50.0% | lr=0.000024
Epoch  10/10 | train_loss=0.5639 train_acc=72.9% | val_loss=1.9035 val_acc=50.0% | lr=0.000000

--- Evaluando Pipeline 32/32 ---
Filtros Raw: Not

c:\Users\Miguel\Documents\GitHub\TFG-BCI\src\DataProvider\FifDataProvider.py:95: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  combined_epochs = mne.concatenate_epochs(all_epochs_list)
c:\Users\Miguel\Documents\GitHub\TFG-BCI\src\pretrainedModels\MiRepNet\utils\utils.py:75: ComplexWarning: Casting complex values to real discards the imaginary part
  XEA[i] = np.dot(sqrtRefEA, x[i])


Epoch   1/10 | train_loss=0.8085 train_acc=44.8% | val_loss=0.7026 val_acc=50.0% | lr=0.000976
Epoch   2/10 | train_loss=0.6962 train_acc=53.1% | val_loss=0.7526 val_acc=50.0% | lr=0.000905
Epoch   3/10 | train_loss=0.6922 train_acc=53.1% | val_loss=0.7205 val_acc=50.0% | lr=0.000794
Epoch   4/10 | train_loss=0.6614 train_acc=63.5% | val_loss=0.7007 val_acc=50.0% | lr=0.000655
Epoch   5/10 | train_loss=0.6749 train_acc=61.5% | val_loss=0.7079 val_acc=50.0% | lr=0.000500
Epoch   6/10 | train_loss=0.6652 train_acc=58.3% | val_loss=0.7229 val_acc=50.0% | lr=0.000345
Epoch   7/10 | train_loss=0.6363 train_acc=66.7% | val_loss=0.7406 val_acc=50.0% | lr=0.000206
Epoch   8/10 | train_loss=0.6376 train_acc=68.8% | val_loss=0.7567 val_acc=50.0% | lr=0.000095
Epoch   9/10 | train_loss=0.6409 train_acc=65.6% | val_loss=0.7638 val_acc=50.0% | lr=0.000024
Epoch  10/10 | train_loss=0.6450 train_acc=62.5% | val_loss=0.7647 val_acc=50.0% | lr=0.000000


,Filtros Usados,Accuracy,Precision,Recall,F1-Score
0,BandpassFilter + Resampler + ICAProcessor + An...,0.875000,0.877622,0.875000,0.874783
1,NotchFilter + BandpassFilter + AnnotationRenamer,0.875000,0.877622,0.875000,0.874783
2,BandpassFilter + ICAProcessor + AnnotationRenamer,0.875000,0.877622,0.875000,0.874783
3,BandpassFilter + Resampler + AnnotationRenamer,0.875000,0.877622,0.875000,0.874783
4,NotchFilter + BandpassFilter + ICAProcessor + ...,0.875000,0.877622,0.875000,0.874783
5,NotchFilter + BandpassFilter + Resampler + Ann...,0.875000,0.877622,0.875000,0.874783
6,NotchFilter + BandpassFilter + Resampler + ICA...,0.875000,0.877622,0.875000,0.874783
7,BandpassFilter + AnnotationRenamer,0.875000,0.877622,0.875000,0.874783
8,NotchFilter + BandpassFilter + CARReference + ...,0.500000,0.250000,0.500000,0.333333
9,NotchFilter + BandpassFilter + CARReference + ...,0.500000,0.250000,0.500000,0.333333


In [6]:
import os

# Creamos una carpeta 'resultados' en tu proyecto si no existe
carpeta_resultados = os.path.join(PROJECT_ROOT, "resultados_experimentos")
os.makedirs(carpeta_resultados, exist_ok=True)

# 1. Guardar como CSV (ideal para programar o abrir en bloc de notas)
ruta_csv = os.path.join(carpeta_resultados, "comparativa_filtros_raw.csv")
df_resultados.to_csv(ruta_csv, index=False, sep=';', decimal=',')
print(f"✅ Resultados guardados en CSV: {ruta_csv}")

# 2. Guardar como Excel (ideal para tu TFG, puedes hacer gráficos allí fácilmente)
# Nota: Si te da error, puede que necesites instalar openpyxl: pip install openpyxl
ruta_excel = os.path.join(carpeta_resultados, "comparativa_filtros_raw.xlsx")
try:
    df_resultados.to_excel(ruta_excel, index=False)
    print(f"✅ Resultados guardados en Excel: {ruta_excel}")
except ModuleNotFoundError:
    print("❌ No se pudo guardar en Excel. Si quieres hacerlo, ejecuta: !pip install openpyxl")

✅ Resultados guardados en CSV: c:\Users\Miguel\Documents\GitHub\TFG-BCI\resultados_experimentos\comparativa_filtros_raw.csv
❌ No se pudo guardar en Excel. Si quieres hacerlo, ejecuta: !pip install openpyxl
